# Leaf YOLO Training on Colab A100

This notebook clones the GitHub repository, prepares a one-class `leaf` dataset, trains YOLO26 candidates, evaluates on the test split, and exports ONNX/TF.js artifacts for browser deployment.

## 1. Clone Repo and Install Dependencies

In [ ]:
%cd /content
!rm -rf leaf-object-detection
!git clone https://github.com/fishman7337/leaf-object-detection.git
%cd /content/leaf-object-detection
!pip install -U -r requirements-colab.txt

## 2. Download Dataset

Upload your Kaggle API token (`kaggle.json`) when prompted. If you do not want to use Kaggle, upload `archive.zip` manually into `/content/leaf-object-detection` and skip this cell.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload kaggle.json

if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    !pip install -U kaggle
    !kaggle datasets download -d sebastianpalaciob/plantvillage-for-object-detection-yolo -p . --force
    !mv plantvillage-for-object-detection-yolo.zip archive.zip
else:
    print('No kaggle.json uploaded. Upload archive.zip manually into /content/leaf-object-detection before continuing.')

## 3. Add Mixed-Scene Public Data and Hard Negatives

In [ ]:
!mkdir -p public
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset public/PlantDoc-Object-Detection-Dataset || true
!python scripts/import_plantdoc_git.py --repo public/PlantDoc-Object-Detection-Dataset --out public/plantdoc_yolo --force
!python scripts/generate_synthetic_negatives.py --out hard_negatives/synthetic --count 300 --size 320

## 4. Prepare and Validate Single-Class Leaf Dataset

In [ ]:
!python scripts/prepare_leaf_dataset.py \
  --archive archive.zip \
  --work-dir . \
  --extra-yolo-dir public/plantdoc_yolo \
  --hard-negatives-dir hard_negatives/synthetic \
  --force

!python scripts/validate_leaf_dataset.py \
  --dataset datasets/leaf_yolo \
  --write-report

## 5. Smoke Test

In [ ]:
!python scripts/train_leaf_yolo.py \
  --data datasets/leaf_yolo/data.yaml \
  --project runs/leaf_yolo \
  --device 0 \
  --smoke \
  --export

## 6. Main Training

Run this after the smoke test succeeds. For a shorter first pass, change `--models` to `yolo26s.pt yolo26m.pt`.

In [ ]:
!python scripts/train_leaf_yolo.py \
  --data datasets/leaf_yolo/data.yaml \
  --project runs/leaf_yolo \
  --device 0 \
  --models yolo26s.pt yolo26m.pt yolo26x.pt \
  --epochs 180 \
  --imgsz 640 \
  --export

## 7. Save Outputs

In [ ]:
!zip -r /content/leaf_yolo_runs.zip runs/leaf_yolo
from google.colab import files
files.download('/content/leaf_yolo_runs.zip')